# Image Caption Generator — COCO Training Notebook (Kaggle)

Same training as `notebooks/train_colab_coco.ipynb` (ResNet encoder + Bahdanau attention + LSTM decoder on a COCO subset via the Karpathy split), adapted for **Kaggle Notebooks** instead of Colab.

**Why Kaggle instead of Colab**: Colab's free tier has undocumented, fluctuating session limits (Google's own words: usage limits "vary over time" and are intentionally not published). In practice this meant sessions getting reclaimed before a single epoch finished on this dataset size. Kaggle's free GPU tier publishes an actual quota -- **30 GPU-hours/week, up to 12 hours per session** -- and, importantly, supports **background execution**: click "Save Version" → "Save & Run All (Commit)" and it keeps running even after you close the browser tab, rather than needing the tab open and connected the whole time.

**Before running:**
1. Notebook Settings (right sidebar) → **Accelerator** → GPU T4 x2 (or P100)
2. Notebook Settings → **Internet** → **On** (off by default; needed for cloning the repo, pip installs, and downloading images)

**Recommended way to run this**: rather than executing cells interactively one at a time (which requires staying connected), use **Save Version → Save & Run All (Commit)** from the top-right menu once you've set the accelerator and internet toggle. This runs the whole notebook top to bottom in the background; you can close the tab and check back later. Outputs (including the trained checkpoint) will be available under the notebook's **Output** tab once the commit finishes, or earlier via `/kaggle/working/` if you're watching an interactive session.

Steps:
1. Clone the repo and install dependencies
2. Get a COCO subset via the Karpathy split (Hugging Face `yerevann/coco-karpathy`)
3. Train (with the same resumable-checkpoint logic as the Colab version, as a safety net if a session does end early)
4. Evaluate (BLEU / METEOR / CIDEr) -- compare against the Flickr8k and Colab-COCO numbers in the README
5. Re-run the out-of-distribution storefront-style hallucination test

## 1. Get the code onto Kaggle

In [ ]:
# Repo is private -- clone with a token (generate one at
# github.com -> Settings -> Developer settings -> Personal access tokens,
# 'repo' scope; revoke it once the clone succeeds, it's not needed again).
# Requires Settings -> Internet -> On in this notebook's right sidebar.
!git clone https://<your-github-token>@github.com/SovanDaraP02/image-caption-generator.git
%cd image-caption-generator

!pwd
!ls

In [ ]:
!pip install -q -e .
!pip install -q datasets

In [ ]:
import os
import sys
# pip install -e . updates site-packages on disk, but this already-running
# kernel cached its module search path at startup, so it won't see the
# newly-installed package until the kernel restarts. Rather than restart
# (which would lose any state from cells run before this point), just add
# the source directory to sys.path directly -- every cell below that
# imports caption_generator relies on this having run first.
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (set Accelerator to GPU in Notebook Settings!)")

## 2. Download a COCO subset (Karpathy split)

Same logic as the Colab notebook, minus the Drive-caching step -- Kaggle's
12-hour session is normally enough to finish in one sitting, and
`/kaggle/working/` persists for the lifetime of this session/commit
regardless.

In [ ]:
# Verified working source as of writing: yerevann/coco-karpathy on the Hub.
# It exposes the Karpathy partition as four separate loadable HF splits
# (train / restval / validation / test). It's text/metadata only
# (captions + an image URL per row); images are fetched separately from
# the official COCO image server in the next cell.
from datasets import load_dataset

for split in ["train", "restval", "validation", "test"]:
    ds = load_dataset("yerevann/coco-karpathy", split=split, streaming=True)
    first = next(iter(ds))
    print(f"{split:12s} -> internal split label: {first['split']:8s}  e.g. {first['filename']}")

In [ ]:
# Subset sizes -- tune down for a faster run, or up for more diversity
# (full Karpathy train split is ~113k images).
N_TRAIN = 50_000
N_VAL = 3_000
N_TEST = 3_000

IMAGE_DIR = "/kaggle/working/data/coco/Images"
import os
os.makedirs(IMAGE_DIR, exist_ok=True)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests


def collect_split_examples(hf_splits, n):
    """hf_splits: list of HF split names to pull from, in order, stopping
    once n examples are collected. For 'train' pass ["train", "restval"]
    since 'restval' is COCO's extra pool the Karpathy split adds to train
    in most published setups."""
    examples = []
    for hf_split in hf_splits:
        if len(examples) >= n:
            break
        stream = load_dataset("yerevann/coco-karpathy", split=hf_split, streaming=True)
        for ex in stream:
            examples.append(ex)
            if len(examples) >= n:
                break
    return examples


def download_images(examples, max_workers=100, max_retries=2):
    """Downloads to IMAGE_DIR, returns (image_filename, caption) pairs --
    one pair per caption, matching Flickr8k's 5-captions-per-image density.
    Skips any file that already exists, so safe to re-run."""
    def fetch(ex):
        path = os.path.join(IMAGE_DIR, ex["filename"])
        if os.path.exists(path):
            return ex, True
        for attempt in range(max_retries + 1):
            try:
                r = requests.get(ex["url"], timeout=10)
                r.raise_for_status()
                with open(path, "wb") as f:
                    f.write(r.content)
                return ex, True
            except Exception:
                if attempt == max_retries:
                    return ex, False
                time.sleep(0.5)

    pairs = []
    failed = 0
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(fetch, ex) for ex in examples]
        for i, future in enumerate(as_completed(futures), 1):
            ex, ok = future.result()
            if ok:
                for cap in ex["sentences"]:
                    pairs.append((ex["filename"], cap))
            else:
                failed += 1
            if i % 2000 == 0:
                elapsed = time.time() - start
                print(f"  {i}/{len(examples)} images ({i/elapsed:.1f} img/s), {failed} failed")
    print(f"Done: {len(examples) - failed}/{len(examples)} images, {len(pairs)} (image, caption) pairs, {failed} failed")
    return pairs


print("Collecting split assignments...")
train_examples = collect_split_examples(["train", "restval"], N_TRAIN)
val_examples = collect_split_examples(["validation"], N_VAL)
test_examples = collect_split_examples(["test"], N_TEST)
print(f"train: {len(train_examples)}  val: {len(val_examples)}  test: {len(test_examples)}")

print("Downloading train images...")
TRAIN_PAIRS = download_images(train_examples)
print("Downloading val images...")
VAL_PAIRS = download_images(val_examples)
print("Downloading test images...")
TEST_PAIRS = download_images(test_examples)

## 3. Train

Same `train_one_epoch`/`validate` as the other notebooks. Still saves a
resumable `latest_checkpoint_coco.pth` after every epoch as a safety net
-- Kaggle's 12-hour session should be enough for this to finish in one
sitting, but if it doesn't, re-running this cell in a fresh session
(after adding the previous session's output as an input dataset via
Kaggle's "Add Data" -> "Your Datasets/Notebooks" picker) will resume
from the last completed epoch instead of starting over.

In [ ]:
import os

import torch.nn as nn
from torch.utils.data import DataLoader

from caption_generator.data.dataset import ImageCaptionDataset, collate_fn
from caption_generator.data.vocabulary import Vocabulary
from caption_generator.models.decoder import DecoderWithAttention
from caption_generator.models.encoder import EncoderCNN
from caption_generator.train import train_one_epoch, validate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CHECKPOINT_PATH = "/kaggle/working/best_checkpoint_coco.pth"
LATEST_PATH = "/kaggle/working/latest_checkpoint_coco.pth"
# If you added a previous session's output as an input dataset to resume,
# point this at its latest_checkpoint_coco.pth (Kaggle mounts input
# datasets read-only under /kaggle/input/<dataset-name>/).
RESUME_FROM_INPUT = None  # e.g. "/kaggle/input/my-coco-run-v1/latest_checkpoint_coco.pth"

TRAIN_CAPTIONS_RAW = [cap for _, cap in TRAIN_PAIRS]
vocab = Vocabulary(min_word_freq=5).build(TRAIN_CAPTIONS_RAW)
pad_idx = vocab.word2idx[vocab.PAD_TOKEN]
print(f"Vocab size: {len(vocab)}")

train_dataset = ImageCaptionDataset(IMAGE_DIR, TRAIN_PAIRS, vocab, split="train")
val_dataset = ImageCaptionDataset(IMAGE_DIR, VAL_PAIRS, vocab, split="val")

BATCH_SIZE = 64  # encoder is frozen, so memory footprint is modest -- plenty of T4 headroom for this
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)

encoder = EncoderCNN(fine_tune=False).to(device)
decoder = DecoderWithAttention(vocab_size=len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(decoder.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

NUM_EPOCHS = 10
EARLY_STOP_PATIENCE = 3
best_val_loss = float("inf")
epochs_without_improvement = 0
start_epoch = 1

resume_source = LATEST_PATH if os.path.exists(LATEST_PATH) else RESUME_FROM_INPUT
if resume_source and os.path.exists(resume_source):
    print(f"Found a previous run's checkpoint at {resume_source} -- resuming instead of starting over.")
    ckpt = torch.load(resume_source, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state"])
    decoder.load_state_dict(ckpt["decoder_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    vocab.word2idx = ckpt["vocab_word2idx"]
    vocab.idx2word = ckpt["vocab_idx2word"]
    pad_idx = vocab.word2idx[vocab.PAD_TOKEN]
    best_val_loss = ckpt["best_val_loss"]
    epochs_without_improvement = ckpt["epochs_without_improvement"]
    start_epoch = ckpt["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}, best_val_loss so far = {best_val_loss:.4f}")
else:
    print("No previous checkpoint found -- starting fresh.")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion, device, pad_idx)
    val_loss = validate(encoder, decoder, val_loader, criterion, device)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  lr={current_lr:.2e}")

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    torch.save({
        "encoder_state": encoder.state_dict(),
        "decoder_state": decoder.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "vocab_word2idx": vocab.word2idx,
        "vocab_idx2word": vocab.idx2word,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
    }, LATEST_PATH)

    if improved:
        torch.save({
            "encoder_state": encoder.state_dict(),
            "decoder_state": decoder.state_dict(),
            "vocab_word2idx": vocab.word2idx,
            "vocab_idx2word": vocab.idx2word,
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"No val_loss improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early.")
        break

print("\nIf this session ends before training finishes: commit this notebook "
      "(Save Version) so /kaggle/working/latest_checkpoint_coco.pth becomes "
      "a downloadable output, then start a fresh session, add that output "
      "as an input dataset, and set RESUME_FROM_INPUT to its path above.")

## 4. Evaluate on the test split (BLEU / METEOR / CIDEr)

In [ ]:
!pip install -q pycocoevalcap

from collections import defaultdict

from caption_generator.evaluate import evaluate

test_pairs_by_image = defaultdict(list)
for fname, cap in TEST_PAIRS:
    test_pairs_by_image[fname].append(cap)

scores = evaluate(CHECKPOINT_PATH, dict(test_pairs_by_image), IMAGE_DIR, device=device)
for metric, value in scores.items():
    print(f"{metric}: {value:.4f}")

print("\nCompare against the Flickr8k and Colab-COCO runs' numbers in README.md's Results table.")

## 5. Re-test the hallucination case

Upload the same kind of people-free photo that the Flickr8k model got
wrong (Kaggle: use "Add Data" -> "Upload" to bring in a file, or point
at any downloaded test image) to see whether this model still
hallucinates a person.

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image

from caption_generator.models.caption_model import CaptionModel

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
vocab = Vocabulary()
vocab.word2idx = checkpoint["vocab_word2idx"]
vocab.idx2word = checkpoint["vocab_idx2word"]

encoder = EncoderCNN(fine_tune=False)
encoder.load_state_dict(checkpoint["encoder_state"])
decoder = DecoderWithAttention(vocab_size=len(vocab))
decoder.load_state_dict(checkpoint["decoder_state"])
model = CaptionModel(encoder, decoder, vocab, device=device)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# CHANGE THIS to an uploaded file's path, or leave as a sample test image
TEST_IMAGE_PATH = os.path.join(IMAGE_DIR, TEST_PAIRS[0][0])

image = Image.open(TEST_IMAGE_PATH).convert("RGB")
image_tensor = transform(image).unsqueeze(0)

caption_beam = model.generate_beam(image_tensor, beam_width=3)
print(f"Caption: {caption_beam}")

plt.imshow(image)
plt.title(caption_beam)
plt.axis("off")
plt.show()

## 6. Getting the checkpoint off Kaggle

`best_checkpoint_coco.pth` is already in `/kaggle/working/`, which
means it's automatically included in this notebook's **Output** tab
once you commit (Save Version -> Save & Run All). From there you can
download it directly, no Drive/token dance needed.